In [1]:
import pandas as pd
import nltk
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
import textstat

# Download required NLTK data
nltk.download('wordnet')
nltk.download('omw-1.4')

# Function to compute scores
def compute_scores(ground_truth, summary):
    # Tokenize the inputs
    ground_truth_tokens = ground_truth.split()
    summary_tokens = summary.split()

    # Compute BLEU score
    smoothing_function = SmoothingFunction().method4
    bleu_score = sentence_bleu([ground_truth_tokens], summary_tokens, smoothing_function=smoothing_function)
    
    # Compute METEOR score
    meteor = meteor_score([ground_truth_tokens], summary_tokens)
    
    # Compute ROUGE scores
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores = scorer.score(ground_truth, summary)
    rouge1 = scores['rouge1'].fmeasure
    rouge2 = scores['rouge2'].fmeasure
    rougeL = scores['rougeL'].fmeasure
    
    # Compute Flesch Reading Ease score
    flesch_reading_ease = textstat.flesch_reading_ease(summary)
    
    return bleu_score, meteor, rouge1, rouge2, rougeL, flesch_reading_ease

def main(input_csv, ground_truth_text, output_csv):
    # Read input CSV
    df = pd.read_csv(input_csv)
    
    # Initialize lists to store scores for BART summaries
    bart_bleu_scores = []
    bart_meteor_scores = []
    bart_rouge1_scores = []
    bart_rouge2_scores = []
    bart_rougeL_scores = []
    bart_flesch_reading_ease_scores = []

    # Initialize lists to store scores for BERT summaries
    bert_bleu_scores = []
    bert_meteor_scores = []
    bert_rouge1_scores = []
    bert_rouge2_scores = []
    bert_rougeL_scores = []
    bert_flesch_reading_ease_scores = []
    
    # Compute scores for BART summaries
    for summary in df['Summary (BART) Reduced']:
        bleu, meteor, rouge1, rouge2, rougeL, flesch_reading_ease = compute_scores(ground_truth_text, summary)
        bart_bleu_scores.append(bleu)
        bart_meteor_scores.append(meteor)
        bart_rouge1_scores.append(rouge1)
        bart_rouge2_scores.append(rouge2)
        bart_rougeL_scores.append(rougeL)
        bart_flesch_reading_ease_scores.append(flesch_reading_ease)

    # Compute scores for BERT summaries
    for summary in df['Summary (BERT) Reduced']:
        bleu, meteor, rouge1, rouge2, rougeL, flesch_reading_ease = compute_scores(ground_truth_text, summary)
        bert_bleu_scores.append(bleu)
        bert_meteor_scores.append(meteor)
        bert_rouge1_scores.append(rouge1)
        bert_rouge2_scores.append(rouge2)
        bert_rougeL_scores.append(rougeL)
        bert_flesch_reading_ease_scores.append(flesch_reading_ease)
    
    # Add scores to DataFrame
    df['BLEU (BART)'] = bart_bleu_scores
    df['METEOR (BART)'] = bart_meteor_scores
    df['ROUGE-1 (BART)'] = bart_rouge1_scores
    df['ROUGE-2 (BART)'] = bart_rouge2_scores
    df['ROUGE-L (BART)'] = bart_rougeL_scores
    df['Flesch Reading Ease (BART)'] = bart_flesch_reading_ease_scores

    df['BLEU (BERT)'] = bert_bleu_scores
    df['METEOR (BERT)'] = bert_meteor_scores
    df['ROUGE-1 (BERT)'] = bert_rouge1_scores
    df['ROUGE-2 (BERT)'] = bert_rouge2_scores
    df['ROUGE-L (BERT)'] = bert_rougeL_scores
    df['Flesch Reading Ease (BERT)'] = bert_flesch_reading_ease_scores
    
    # Save DataFrame to new CSV
    df.to_csv(output_csv, index=False)
    
if __name__ == "__main__":
    input_csv = 'CSV/Refining/UnicodeR1Janus.csv'  
    ground_truth_text = """ The agent-based model used in this study aims to investigate the dynamics of household behaviors in a rangeland grazing system. The model examines how different behavioral strategies, including risk-taking, and social learning, impact livestock management and resource sustainability . The goal is to understand the long-term social-ecological performance of the system. Key parameters set in the model include the number of households at 60, the initial number of livestock per household at 90, and the simulation running for 100 time steps . The model assigns 20 households to each type of learning strategy: no learning, learning by doing, and social learning. Additional parameters include a knowledge radius of 1 out of 5 (determining the extent of the local neighborhood around each household where it can observe and interact with the environment and other households),  the inclusion of risk mode (which represents various factors influencing an agent’s disposition to act in a given way under uncertainty), and social learning strategy switch (agents can adapt their behavior based on the observed success of their neighbors).

At the beginning, household risk attitudes are diverse, spread from -0.15 to 0.15.  By time step 20, higher risk adoption peaks but  stabilizes and declines by time step 40, indicating a shift toward risk-averse behaviors.

The reserve biomass starts at 90,000 units, drops by time step 30 , then stabilizes around 120,000 units by time step 60. Green biomass mirrors this trend, starting at 45,000 units and stabilizing at 60,000 units after an initial decline. Furthermore, mean livestock and destock values follow a similar pattern, starting at 90, dipping by time step 30, and stabilizing at 50 units by time step 60. Destock values begin at 7, drop to 2 by time step 20, and stabilize around 1 unit by time step 40, with cumulative destock values rising steadily to time step 100.

The Gini coefficient for healthy livestock, initially near zero, rapidly increases and stabilizes by time step 20, indicating growing inequality in livestock distribution. The number of households who operate under a social learning strategy rises rapidly until time step 20 and then stabilizes over the next 20 steps. In contrast, there is significant fluctuation in the number of households who do not learn time step 20, stabilizing with fluctuations by time step 60. 
"""
    output_csv = 'CSV/Refining/ScoresR1Janus.csv'
    main(input_csv, ground_truth_text, output_csv)


[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/noeflandre/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/noeflandre/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
